# 08 - Robust Recovery

Multi-sweep iterative 1D recovery, blind candidates, noise robustness, and `N_CW` scaling under realistic stochastic noise.

In [1]:
from pathlib import Path
import sys
import time
import copy
import importlib
import concurrent.futures as cf

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd()
sys.path.insert(0, str((HERE / "../CW_lnL_check").resolve()))

import cw_helpers
importlib.reload(cw_helpers)

from cw_helpers import (
    build_disco_likelihood,
    build_fast_scan_likelihood,
    build_enterprise_pta,
    compute_mode_spacing,
    generate_injection_params,
    load_pulsars,
    make_distance_optimizer,
    scan_pulsar_distance,
    simulate,
)

# Main setup
N_PSR = 40
N_CW = 4
MAX_N_CW = 16
LOG10_H = -12.0
LOG10_MC = None
COMPONENTS = 30
GWB_LOG10_A = -14.5
GWB_GAMMA = 13 / 3
INCLUDE_GWB = True
INCLUDE_RN = True
RN_COMPONENTS = 30
STOCHASTIC_SCENARIO = "well_separated"
TRUTH_DISTANCE_MODE = "gaussian_prior_draw"

# Seeds
RNG_SEED = 12345
NOISE_SEEDS = [24680, 13579, 99999, 54321, 11111, 77777, 42424, 88888]

# Multi-sweep settings
N_SWEEPS = 5
SWEEP_SCAN_POINTS = 2000
LBFGS_POLISH_MAXITER = 50
LBFGS_FACTR = 1e7

# Blind candidate settings
MODE_PRIOR_SIGMA_WIDTH = 3.0
TRUTH_SIGMA_CLIP = 3.0
TRUTH_SIGMA_MULTIPLIER = 1.0
N_CONTEXT_RANDOM = 3
MAX_1D_SCAN_POINTS = 3000
MODE_SAMPLES_PER_MODE = 6
TOP_MODE_CANDIDATES = 200
PARALLEL_CANDIDATE_SCANS = True
MAX_CANDIDATE_WORKERS = 4
GRADIENT_COMPARE_STARTS = 6

# N_CW scaling
N_CW_VALUES = [2, 4, 6, 8, 12, 16]
SCALING_NOISE_SEED = NOISE_SEEDS[0]

# Gates
RUN_VALIDATION = True
RUN_NOISE_ROBUSTNESS = True
RUN_NCW_SCALING = True
MAKE_PLOTS = False
USE_FAST_LIKELIHOOD = True

print(HERE)

/home/mattm/miniforge3/envs/discotech/lib/python3.12/site-packages/enterprise/signals/utils.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename


/home/mattm/projects/HSYMT/lnL_distance_scans


## Helpers

In [2]:
CW_LIBRARY = [
    dict(cos_gwtheta=0.30, gwphi=2.50, cos_inc=-0.20, phase0=1.00, psi=0.70, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.00),
    dict(cos_gwtheta=-0.50, gwphi=0.80, cos_inc=0.40, phase0=2.10, psi=1.30, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.80),
    dict(cos_gwtheta=0.05, gwphi=4.20, cos_inc=-0.65, phase0=0.35, psi=2.30, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.20),
    dict(cos_gwtheta=0.70, gwphi=5.40, cos_inc=0.10, phase0=1.70, psi=0.20, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.90),
    dict(cos_gwtheta=-0.10, gwphi=3.30, cos_inc=0.80, phase0=2.90, psi=1.80, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.10),
    dict(cos_gwtheta=0.45, gwphi=1.60, cos_inc=-0.35, phase0=0.80, psi=2.70, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.70),
    dict(cos_gwtheta=-0.75, gwphi=5.90, cos_inc=0.55, phase0=2.40, psi=0.45, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.35),
    dict(cos_gwtheta=0.18, gwphi=0.25, cos_inc=-0.85, phase0=1.35, psi=1.05, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.60),
    dict(cos_gwtheta=-0.32, gwphi=2.05, cos_inc=0.25, phase0=3.05, psi=2.05, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.45),
    dict(cos_gwtheta=0.88, gwphi=3.75, cos_inc=-0.05, phase0=0.15, psi=0.95, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.95),
    dict(cos_gwtheta=-0.62, gwphi=4.75, cos_inc=0.72, phase0=1.95, psi=2.85, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.25),
    dict(cos_gwtheta=0.58, gwphi=0.95, cos_inc=-0.48, phase0=2.75, psi=1.55, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.75),
    dict(cos_gwtheta=-0.18, gwphi=5.15, cos_inc=0.08, phase0=0.55, psi=0.10, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.05),
    dict(cos_gwtheta=0.02, gwphi=1.20, cos_inc=-0.70, phase0=2.25, psi=2.45, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.85),
    dict(cos_gwtheta=-0.88, gwphi=3.95, cos_inc=0.38, phase0=1.15, psi=1.75, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.30),
    dict(cos_gwtheta=0.36, gwphi=4.55, cos_inc=-0.18, phase0=2.55, psi=0.60, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.65),
]


def clipped_normal(mean, sigma, rng, clip):
    draw = rng.normal(mean, sigma)
    lo = np.maximum(0.01, mean - clip * sigma)
    hi = mean + clip * sigma
    return np.clip(draw, lo, hi)


def select_pulsars(n_psr):
    ent_all, disco_all = load_pulsars(None)
    pairs = sorted(zip(ent_all, disco_all), key=lambda pair: pair[1].pdist[1])
    selected = pairs[:n_psr]
    return [p[0] for p in selected], [p[1] for p in selected], len(disco_all)


def clone_psrs_with_distances(psrs, distances):
    clones = copy.deepcopy(psrs)
    for psr, dist in zip(clones, distances):
        sigma = float(psr.pdist[1])
        try:
            psr.pdist = (float(dist), sigma)
        except Exception:
            psr._pdist = (float(dist), sigma)
    return clones


def make_truth_distances(prior_mean, prior_sigma, rng):
    if TRUTH_DISTANCE_MODE == "gaussian_prior_draw":
        return clipped_normal(prior_mean, prior_sigma * TRUTH_SIGMA_MULTIPLIER, rng, TRUTH_SIGMA_CLIP)
    if TRUTH_DISTANCE_MODE == "prior_mean":
        return prior_mean.copy()
    raise ValueError(TRUTH_DISTANCE_MODE)


def cw_params_from_library(n_cw, log10_h=None, log10_mc=None):
    if n_cw > len(CW_LIBRARY):
        raise ValueError(f"N_CW={n_cw} exceeds CW_LIBRARY size={len(CW_LIBRARY)}")
    out = [dict(cw) for cw in CW_LIBRARY[:n_cw]]
    for cw in out:
        if log10_h is not None:
            cw["log10_h"] = log10_h
        if log10_mc is not None:
            cw["log10_mc"] = log10_mc
    return out


def cw_list_from_enterprise_params(enterprise_params, cw_block_names):
    keys = ("cos_gwtheta", "gwphi", "cos_inc", "log10_mc", "log10_fgw", "log10_h", "phase0", "psi")
    return [{key: float(enterprise_params[f"{name}_{key}"]) for key in keys} for name in cw_block_names]


def silence_extra_cws(params, cw_block_names, active_n_cw):
    out = dict(params)
    for idx, name in enumerate(cw_block_names):
        if idx >= active_n_cw:
            out[f"{name}_log10_h"] = -50.0
    return out


def distance_key(psr):
    return f"{psr.name}_cw_p_dist"


def set_distances(values, param_keys, disco_psrs, distances):
    out = np.array(values, dtype=float).copy()
    for psr, dist in zip(disco_psrs, distances):
        out[param_keys.index(distance_key(psr))] = dist
    return out


def min_mode_spacings(disco_psrs, cw_params_list):
    out = []
    for psr in disco_psrs:
        vals = [compute_mode_spacing(cw["cos_gwtheta"], cw["gwphi"], cw["log10_fgw"], psr.pos) for cw in cw_params_list]
        out.append(float(np.nanmin(vals)))
    return np.array(out)


def score_distances(case, distances):
    err = np.asarray(distances, dtype=float) - case["truth_dist"]
    em = err / case["mode_spacings"]
    es = err / case["sigmas"]
    return dict(
        n_recovered=int(np.sum(np.abs(em) < 0.5)),
        frac_recovered=float(np.mean(np.abs(em) < 0.5)),
        median_abs_modes=float(np.nanmedian(np.abs(em))),
        max_abs_modes=float(np.nanmax(np.abs(em))),
        median_abs_sigma=float(np.nanmedian(np.abs(es))),
        max_abs_sigma=float(np.nanmax(np.abs(es))),
    )


def top_local_maxima(x, y, k):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(y)
    x, y = x[finite], y[finite]
    if len(x) == 0:
        return np.array([]), np.array([])
    if len(x) >= 3:
        idx = np.where(np.r_[False, (y[1:-1] >= y[:-2]) & (y[1:-1] >= y[2:]), False])[0]
    else:
        idx = np.array([], dtype=int)
    if len(idx) == 0:
        idx = np.array([int(np.nanargmax(y))])
    idx = idx[np.argsort(y[idx])[::-1]][:k]
    return x[idx], y[idx]


def scan_bounds(case, i, n_points=None):
    lo = max(0.01, case["prior_mean"][i] - MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i])
    hi = case["prior_mean"][i] + MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i]
    if n_points is not None:
        return lo, hi, int(n_points)
    dL = case["mode_spacings"][i]
    if np.isfinite(dL) and dL > 0:
        n = int(np.ceil((hi - lo) / dL * MODE_SAMPLES_PER_MODE)) + 1
    else:
        n = MAX_1D_SCAN_POINTS
    return lo, hi, int(np.clip(n, 25, MAX_1D_SCAN_POINTS))


def conditional_scan(case, i, context_distances, n_points=SWEEP_SCAN_POINTS, required_points=None):
    lo, hi, n = scan_bounds(case, i, n_points)
    req = [case["prior_mean"][i]]
    if required_points is not None:
        req += list(np.ravel(required_points))
    vals = set_distances(case["base_values"], case["param_keys"], case["disco_psrs"], context_distances)
    x, y = scan_pulsar_distance(
        case["logl_fn"], vals, case["param_keys"], distance_key(case["disco_psrs"][i]),
        lo, hi, n_points=n, chunk_size=None, n_components=case["components"],
        required_points=req,
    )
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float)


def scan_metrics(case, i, context_distances, n_points=SWEEP_SCAN_POINTS):
    x, y = conditional_scan(case, i, context_distances, n_points=n_points, required_points=[case["truth_dist"][i]])
    truth_idx = int(np.nanargmin(np.abs(x - case["truth_dist"][i])))
    truth_y = float(y[truth_idx])
    local = np.where(np.abs(x - case["truth_dist"][i]) <= 0.5 * case["mode_spacings"][i])[0]
    valley = float(np.nanmin(y[local])) if len(local) else float(np.nanmin(y))
    px, py = top_local_maxima(x, y, TOP_MODE_CANDIDATES)
    return dict(
        contrast=float(truth_y - valley),
        n_degenerate=int(np.sum(py >= truth_y - 1.0)) if len(py) else 0,
        truth_is_global=bool(truth_y >= float(np.nanmax(y)) - 1e-6),
    )


def choose_peak_distance(case, i, context_distances, n_points=SWEEP_SCAN_POINTS):
    x, y = conditional_scan(case, i, context_distances, n_points=n_points)
    px, py = top_local_maxima(x, y, TOP_MODE_CANDIDATES)
    if len(px) == 0:
        j = int(np.nanargmax(y))
        return float(x[j]), float(y[j])
    return float(px[0]), float(py[0])

## Build Infrastructure

In [3]:
def build_case(noise_seed, active_n_cw=N_CW, max_n_cw=None, pta_bundle=None):
    """Build one stochastic likelihood. Same RNG_SEED fixes truth + CW params; noise_seed changes only simulation draw."""
    max_n_cw = active_n_cw if max_n_cw is None else max_n_cw
    t0 = time.time()
    rng = np.random.default_rng(RNG_SEED)

    if pta_bundle is None:
        ent_psrs, disco_psrs, total_loaded = select_pulsars(N_PSR)
        prior_mean = np.array([p.pdist[0] for p in disco_psrs], dtype=float)
        sigmas = np.array([p.pdist[1] for p in disco_psrs], dtype=float)
        truth_dist = make_truth_distances(prior_mean, sigmas, rng)
        ent_psrs_inj = clone_psrs_with_distances(ent_psrs, truth_dist)
        pta, cw_block_names, _ = build_enterprise_pta(
            ent_psrs_inj, max_n_cw, components=COMPONENTS,
            include_rn=INCLUDE_RN, rn_components=RN_COMPONENTS,
        )
        enterprise_params_full = generate_injection_params(
            pta, ent_psrs_inj, max_n_cw, cw_block_names,
            log10_h=LOG10_H,
            scenario=STOCHASTIC_SCENARIO,
            rng=rng,
            gwb_log10_A=GWB_LOG10_A,
            gwb_gamma=GWB_GAMMA,
            include_rn=INCLUDE_RN,
        )
        pta_bundle = dict(
            pta=pta, cw_block_names=cw_block_names,
            enterprise_params_full=enterprise_params_full,
            disco_psrs=disco_psrs, prior_mean=prior_mean,
            sigmas=sigmas, truth_dist=truth_dist,
            total_loaded=total_loaded,
        )
    else:
        pta = pta_bundle["pta"]
        cw_block_names = pta_bundle["cw_block_names"]
        enterprise_params_full = pta_bundle["enterprise_params_full"]
        disco_psrs = pta_bundle["disco_psrs"]
        prior_mean = pta_bundle["prior_mean"]
        sigmas = pta_bundle["sigmas"]
        truth_dist = pta_bundle["truth_dist"]
        total_loaded = pta_bundle["total_loaded"]

    enterprise_params = silence_extra_cws(enterprise_params_full, cw_block_names, active_n_cw)
    cw_params_all = cw_list_from_enterprise_params(enterprise_params, cw_block_names)
    cw_params_active = cw_params_all[:active_n_cw]

    np.random.seed(noise_seed)
    sim_resids = simulate(pta, enterprise_params, sparse_cholesky=True)
    residual_map = {getattr(p, "name", p): y for p, y in zip(pta.pulsars, sim_resids)}

    logl_fn_disco, param_keys_disco, base_values_disco = build_disco_likelihood(
        disco_psrs, residual_map,
        num_cw=max_n_cw,
        enterprise_params=enterprise_params,
        cw_block_names=cw_block_names,
        components=COMPONENTS,
        include_gwb=INCLUDE_GWB,
        include_rn=INCLUDE_RN,
        rn_components=RN_COMPONENTS,
    )
    logl_fn, param_keys, base_values = logl_fn_disco, param_keys_disco, np.asarray(base_values_disco, dtype=float)

    if USE_FAST_LIKELIHOOD:
        try:
            logl_fn_fast, param_keys_fast, base_values_fast = build_fast_scan_likelihood(
                disco_psrs, residual_map,
                num_cw=max_n_cw,
                enterprise_params=enterprise_params,
                cw_block_names=cw_block_names,
                components=COMPONENTS,
                include_rn=INCLUDE_RN,
                rn_components=RN_COMPONENTS,
            )
            if param_keys_fast == param_keys_disco:
                truth_probe = set_distances(base_values_fast, param_keys_fast, disco_psrs, truth_dist)
                delta = float(logl_fn_fast(jnp.asarray(truth_probe)) - logl_fn_disco(jnp.asarray(truth_probe)))
                if abs(delta) <= 1e-6:
                    logl_fn, param_keys, base_values = logl_fn_fast, param_keys_fast, np.asarray(base_values_fast, dtype=float)
                else:
                    print(f"  fast validation failed delta={delta:.3e}; using discovery")
            else:
                print("  fast key mismatch; using discovery")
        except Exception as e:
            print(f"  fast likelihood failed ({e}); using discovery")

    mode_spacings = min_mode_spacings(disco_psrs, cw_params_active)
    truth_values = set_distances(base_values, param_keys, disco_psrs, truth_dist)
    prior_values = set_distances(base_values, param_keys, disco_psrs, prior_mean)
    truth_lnL = float(logl_fn(jnp.asarray(truth_values)))
    prior_mean_lnL = float(logl_fn(jnp.asarray(prior_values)))

    case = dict(
        noise_seed=noise_seed, active_n_cw=active_n_cw, max_n_cw=max_n_cw,
        components=COMPONENTS, pta_bundle=pta_bundle, pta=pta,
        cw_block_names=cw_block_names, enterprise_params=enterprise_params,
        disco_psrs=disco_psrs, prior_mean=prior_mean, sigmas=sigmas,
        truth_dist=truth_dist, cw_params_active=cw_params_active,
        logl_fn=logl_fn, param_keys=param_keys, base_values=base_values,
        truth_values=truth_values, prior_values=prior_values,
        mode_spacings=mode_spacings, truth_lnL=truth_lnL,
        prior_mean_lnL=prior_mean_lnL, lnL_gap=truth_lnL - prior_mean_lnL,
        build_seconds=time.time() - t0,
        total_loaded=total_loaded,
    )
    print(
        f"built seed={noise_seed}, N_CW={active_n_cw}, "
        f"lnL_gap={case['lnL_gap']:.3g}, seconds={case['build_seconds']:.1f}"
    )
    return case

## Blind Candidates And Optimizers

In [4]:
def make_candidate_for_pulsar_blind(case, i):
    """Candidates WITHOUT truth context: prior_mean + random prior draws only."""
    context_rng = np.random.default_rng(RNG_SEED + 1000 + i)
    contexts = [case["prior_mean"].copy()]
    for _ in range(N_CONTEXT_RANDOM):
        contexts.append(clipped_normal(case["prior_mean"], case["sigmas"], context_rng, MODE_PRIOR_SIGMA_WIDTH))

    peaks = []
    scores = []
    for ctx in contexts:
        x, y = conditional_scan(case, i, ctx, n_points=None)
        px, py = top_local_maxima(x, y, TOP_MODE_CANDIDATES)
        peaks.extend(px.tolist())
        scores.extend(py.tolist())

    if len(peaks) == 0:
        return np.array([case["prior_mean"][i]]), np.array([0.0])
    df = pd.DataFrame({"distance": peaks, "score": scores})
    dL = case["mode_spacings"][i]
    if np.isfinite(dL) and dL > 0:
        df["mode_bin"] = np.round(df["distance"] / dL).astype(int)
        df = df.sort_values("score", ascending=False).drop_duplicates("mode_bin")
    else:
        df = df.sort_values("score", ascending=False)
    df = df.head(TOP_MODE_CANDIDATES)
    return df["distance"].to_numpy(float), df["score"].to_numpy(float)


def build_empirical_candidates_blind(case):
    print("building blind candidates")
    t0 = time.time()
    n_psr = len(case["disco_psrs"])
    all_modes = [None] * n_psr
    all_scores = [None] * n_psr
    if PARALLEL_CANDIDATE_SCANS:
        with cf.ThreadPoolExecutor(max_workers=MAX_CANDIDATE_WORKERS) as ex:
            futs = {ex.submit(make_candidate_for_pulsar_blind, case, i): i for i in range(n_psr)}
            for n, fut in enumerate(cf.as_completed(futs), 1):
                i = futs[fut]
                all_modes[i], all_scores[i] = fut.result()
                if n % 5 == 0 or n == n_psr:
                    print(f"  candidates {n}/{n_psr}")
    else:
        for i in range(n_psr):
            all_modes[i], all_scores[i] = make_candidate_for_pulsar_blind(case, i)
            if (i + 1) % 5 == 0 or i + 1 == n_psr:
                print(f"  candidates {i + 1}/{n_psr}")
    print(f"candidate seconds={time.time() - t0:.1f}")
    return all_modes, all_scores


def multi_sweep_iterative_1d(case, n_sweeps=N_SWEEPS, scan_points=SWEEP_SCAN_POINTS, verbose=True):
    hardness = case["sigmas"] / case["mode_spacings"]
    order = list(np.argsort(hardness))
    current = case["prior_mean"].copy()
    history = []
    for sweep in range(n_sweeps):
        t0 = time.time()
        changed = 0
        for i in order:
            peak, peak_lnL = choose_peak_distance(case, i, current, n_points=scan_points)
            if abs(peak - current[i]) > 0.1 * case["mode_spacings"][i]:
                changed += 1
            current[i] = peak
        sc = score_distances(case, current)
        dt = time.time() - t0
        history.append(dict(sweep=sweep + 1, seconds=dt, changed=changed, **sc))
        if verbose:
            print(f"  sweep {sweep + 1}: {sc['n_recovered']}/{len(current)} recovered, {changed} changed, {dt:.1f}s")
        if changed == 0:
            break
    return current, pd.DataFrame(history)


def make_polish_optimizer(case, maxiter=LBFGS_POLISH_MAXITER):
    try:
        return make_distance_optimizer(
            case["logl_fn"], case["param_keys"], case["base_values"], case["disco_psrs"],
            prior_means=case["prior_mean"], prior_sigmas=case["sigmas"],
            n_sigma=MODE_PRIOR_SIGMA_WIDTH, objective="logpost",
            maxiter=maxiter, factr=LBFGS_FACTR,
        )[0]
    except TypeError:
        return make_distance_optimizer(
            case["logl_fn"], case["param_keys"], case["base_values"], case["disco_psrs"],
            prior_means=case["prior_mean"], prior_sigmas=case["sigmas"],
            n_sigma=MODE_PRIOR_SIGMA_WIDTH, objective="logpost",
            maxiter=maxiter,
        )[0]


def polish_distances(case, d_start):
    opt = make_polish_optimizer(case)
    d_opt, lnL, logpost, info = opt(d_start)
    return d_opt, float(lnL), float(logpost), info


def weighted_choice(modes, scores, rng):
    modes = np.asarray(modes, dtype=float)
    if len(modes) == 0:
        return np.nan
    scores = np.asarray(scores, dtype=float)
    if len(scores) != len(modes) or not np.all(np.isfinite(scores)):
        return float(rng.choice(modes))
    w = np.exp(scores - np.nanmax(scores))
    w = w / np.sum(w) if np.sum(w) > 0 else np.ones(len(modes)) / len(modes)
    return float(rng.choice(modes, p=w))


def run_best_candidate_gradient(case, candidates, scores, n_starts=GRADIENT_COMPARE_STARTS):
    opt = make_polish_optimizer(case)
    rng = np.random.default_rng(RNG_SEED + case["noise_seed"])
    starts = []
    best_1d = np.array([m[0] if len(m) else case["prior_mean"][i] for i, m in enumerate(candidates)], dtype=float)
    starts.append(("best_1d", best_1d))
    for k in range(max(0, n_starts - 1)):
        d = np.array([
            weighted_choice(candidates[i], scores[i], rng) if len(candidates[i]) else case["prior_mean"][i]
            for i in range(len(candidates))
        ], dtype=float)
        starts.append((f"mode_combo_{k}", d))

    results = []
    for label, d0 in starts:
        d_opt, lnL, logpost, info = opt(d0)
        results.append(dict(label=label, d_opt=d_opt, lnL=float(lnL), logpost=float(logpost), **info, **score_distances(case, d_opt)))
    results.sort(key=lambda r: r["logpost"], reverse=True)
    best = results[0]
    print(f"  gradient compare best={best['label']}: {best['n_recovered']}/{len(candidates)} recovered")
    return best, pd.DataFrame([{k: v for k, v in r.items() if k != "d_opt"} for r in results])

## Quick Validation

In [5]:
validation_case = None
validation_summary = {}

if RUN_VALIDATION:
    print("QUICK VALIDATION")
    validation_case = build_case(NOISE_SEEDS[0], active_n_cw=N_CW)
    print(f"truth_lnL={validation_case['truth_lnL']:.3f}")
    print(f"prior_mean_lnL={validation_case['prior_mean_lnL']:.3f}")
    print(f"lnL_gap={validation_case['lnL_gap']:.3f}")

    d_sweep, sweep_hist = multi_sweep_iterative_1d(validation_case, verbose=True)
    d_polish, lnL_polish, logpost_polish, info_polish = polish_distances(validation_case, d_sweep)
    sc_sweep = score_distances(validation_case, d_sweep)
    sc_polish = score_distances(validation_case, d_polish)
    print(f"sweep validation: {sc_sweep['n_recovered']}/{N_PSR}")
    print(f"polish validation: {sc_polish['n_recovered']}/{N_PSR}, nit={info_polish.get('nit')}")
    display(sweep_hist)
    validation_summary = dict(sweep=sc_sweep, polish=sc_polish, sweep_history=sweep_hist)
else:
    print("Validation skipped")

QUICK VALIDATION
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.r

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built seed=24680, N_CW=4, lnL_gap=1.84e+06, seconds=49.2
truth_lnL=158941.139
prior_mean_lnL=-1679351.629
lnL_gap=1838292.769
  sweep 1: 16/40 recovered, 40 changed, 76.4s
  sweep 2: 18/40 recovered, 14 changed, 9.1s
  sweep 3: 19/40 recovered, 2 changed, 9.0s
  sweep 4: 19/40 recovered, 0 changed, 9.0s
sweep validation: 19/40
polish validation: 19/40, nit=50


,sweep,seconds,changed,n_recovered,frac_recovered,median_abs_modes,max_abs_modes,median_abs_sigma,max_abs_sigma
0,1,76.432175,40,16,0.400,35.150969,930.570038,0.175662,4.077775
1,2,9.088471,14,18,0.450,18.746183,930.570038,0.085343,3.558287
2,3,8.996890,2,19,0.475,15.632922,930.570038,0.062320,3.558287
3,4,9.011977,0,19,0.475,15.632922,930.570038,0.062320,3.558287


## Noise Robustness Loop

In [6]:
noise_results = []
noise_failure_rows = []
noise_gradient_details = {}

if RUN_NOISE_ROBUSTNESS:
    for seed_idx, noise_seed in enumerate(NOISE_SEEDS):
        print("\n" + "=" * 60)
        print(f"NOISE SEED {noise_seed} ({seed_idx + 1}/{len(NOISE_SEEDS)})")
        t_seed = time.time()
        case = build_case(noise_seed, active_n_cw=N_CW)

        candidates, candidate_scores = build_empirical_candidates_blind(case)
        d_sweep, sweep_history = multi_sweep_iterative_1d(case, verbose=True)
        d_polish, lnL_polish, logpost_polish, info_polish = polish_distances(case, d_sweep)
        grad_best, grad_df = run_best_candidate_gradient(case, candidates, candidate_scores)
        noise_gradient_details[noise_seed] = grad_df

        sc_sweep = score_distances(case, d_sweep)
        sc_polish = score_distances(case, d_polish)
        sc_grad = score_distances(case, grad_best["d_opt"])

        err_polish = (d_polish - case["truth_dist"]) / case["mode_spacings"]
        failed = np.where(np.abs(err_polish) >= 0.5)[0].tolist()
        for i in failed:
            noise_failure_rows.append(dict(
                noise_seed=noise_seed, idx=i, pulsar=case["disco_psrs"][i].name,
                sigma_over_mode=float(case["sigmas"][i] / case["mode_spacings"][i]),
                err_modes=float(err_polish[i]),
            ))

        row = dict(
            noise_seed=noise_seed,
            sweep_recovered=sc_sweep["n_recovered"],
            polished_recovered=sc_polish["n_recovered"],
            gradient_recovered=sc_grad["n_recovered"],
            sweep_frac=sc_sweep["frac_recovered"],
            polished_frac=sc_polish["frac_recovered"],
            gradient_frac=sc_grad["frac_recovered"],
            lnL_gap=case["lnL_gap"],
            truth_lnL=case["truth_lnL"],
            prior_mean_lnL=case["prior_mean_lnL"],
            polish_lnL=lnL_polish,
            polish_logpost=logpost_polish,
            polish_nit=info_polish.get("nit"),
            seconds=time.time() - t_seed,
            sweep_history=sweep_history.to_dict("records"),
        )
        noise_results.append(row)
        print(
            f"RESULT seed={noise_seed}: sweep={row['sweep_recovered']}/{N_PSR}, "
            f"polish={row['polished_recovered']}/{N_PSR}, gradient={row['gradient_recovered']}/{N_PSR}, "
            f"seconds={row['seconds']:.1f}"
        )

noise_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("sweep_history",)} for r in noise_results])
noise_failures_df = pd.DataFrame(noise_failure_rows)
display(noise_df)


NOISE SEED 24680 (1/8)
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherP

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built seed=24680, N_CW=4, lnL_gap=1.84e+06, seconds=28.7
building blind candidates


E0507 14:30:31.159093  801838 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0
E0507 14:30:40.529477  801841 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0
E0507 14:30:58.645130  801840 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0
E0507 14:31:00.740709  801841 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0
E0507 14:31:01.346172  801840 pjrt_stream_executor_client.cc:2111] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0
E0507 14:31:01.753369  801841 pjrt_stream_exe

ValueError: RESOURCE_EXHAUSTED: Failed to allocate request for 2.23GiB (2399060208B) on device ordinal 0

## N_CW Scaling Survey

In [ ]:
ncw_results = []

if RUN_NCW_SCALING:
    print("N_CW SCALING")
    # Build max-CW PTA once, then silence extras and rebuild only residual-dependent likelihoods.
    max_case = build_case(SCALING_NOISE_SEED, active_n_cw=MAX_N_CW, max_n_cw=MAX_N_CW)
    pta_bundle = max_case["pta_bundle"]

    for active_n_cw in N_CW_VALUES:
        print("\n" + "-" * 50)
        print(f"N_CW={active_n_cw}")
        t0 = time.time()
        case = build_case(SCALING_NOISE_SEED, active_n_cw=active_n_cw, max_n_cw=MAX_N_CW, pta_bundle=pta_bundle)
        d_sweep, sweep_history = multi_sweep_iterative_1d(case, verbose=True)
        d_polish, lnL_polish, logpost_polish, info_polish = polish_distances(case, d_sweep)
        sc_sweep = score_distances(case, d_sweep)
        sc_polish = score_distances(case, d_polish)

        contrast_idxs = np.linspace(0, N_PSR - 1, 5).round().astype(int)
        contrasts = []
        for i in contrast_idxs:
            m = scan_metrics(case, int(i), case["truth_dist"], n_points=1000)
            contrasts.append(m["contrast"])

        row = dict(
            N_CW=active_n_cw,
            sweep_recovered=sc_sweep["n_recovered"],
            polished_recovered=sc_polish["n_recovered"],
            sweep_frac=sc_sweep["frac_recovered"],
            polished_frac=sc_polish["frac_recovered"],
            contrast_median=float(np.nanmedian(contrasts)),
            lnL_gap=case["lnL_gap"],
            polish_lnL=lnL_polish,
            polish_nit=info_polish.get("nit"),
            seconds=time.time() - t0,
        )
        ncw_results.append(row)
        print(
            f"N_CW={active_n_cw}: sweep={row['sweep_recovered']}/{N_PSR}, "
            f"polish={row['polished_recovered']}/{N_PSR}, "
            f"contrast_med={row['contrast_median']:.3g}, seconds={row['seconds']:.1f}"
        )

ncw_df = pd.DataFrame(ncw_results)
display(ncw_df)

## Summary Tables

In [ ]:
print("NOISE ROBUSTNESS (N_PSR=40, N_CW=4, GWB=-14.5, RN=True)")
print("=" * 57)
if len(noise_df):
    display_cols = [
        "noise_seed", "sweep_recovered", "polished_recovered",
        "gradient_recovered", "lnL_gap", "seconds",
    ]
    display(noise_df[display_cols])
    mean_row = noise_df[["sweep_recovered", "polished_recovered", "gradient_recovered", "seconds"]].mean()
    print(
        f"MEAN: sweep={mean_row['sweep_recovered']:.1f}/{N_PSR}, "
        f"polish={mean_row['polished_recovered']:.1f}/{N_PSR}, "
        f"gradient={mean_row['gradient_recovered']:.1f}/{N_PSR}, "
        f"seconds={mean_row['seconds']:.1f}"
    )
else:
    print("No noise robustness results")

print("\nN_CW SCALING (N_PSR=40, seed=24680, GWB=-14.5, RN=True)")
print("=" * 57)
if len(ncw_df):
    display(ncw_df[["N_CW", "sweep_recovered", "polished_recovered", "contrast_median", "seconds"]])
else:
    print("No N_CW scaling results")

print("\nPER-PULSAR FAILURE ANALYSIS")
print("=" * 27)
if len(noise_failures_df):
    fail_summary = (
        noise_failures_df
        .groupby(["idx", "pulsar", "sigma_over_mode"], as_index=False)
        .agg(
            n_failed=("noise_seed", "count"),
            failed_seeds=("noise_seed", lambda s: list(map(int, s))),
            median_abs_err_modes=("err_modes", lambda s: float(np.median(np.abs(s)))),
        )
        .sort_values(["n_failed", "sigma_over_mode"], ascending=[False, False])
    )
    hard_failures = fail_summary[fail_summary["n_failed"] > len(NOISE_SEEDS) / 2]
    if len(hard_failures):
        display(hard_failures)
    else:
        print("No pulsar failed in >50% of noise seeds.")
        display(fail_summary.head(20))
else:
    print("No polished failures across noise seeds.")

## Optional Plots

In [ ]:
if MAKE_PLOTS:
    if len(noise_df):
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(noise_df["noise_seed"].astype(str), noise_df["sweep_frac"], marker="o", label="multi-sweep")
        ax.plot(noise_df["noise_seed"].astype(str), noise_df["polished_frac"], marker="o", label="polished")
        ax.plot(noise_df["noise_seed"].astype(str), noise_df["gradient_frac"], marker="o", label="candidate gradient")
        ax.set_ylim(0, 1.05)
        ax.set_xlabel("Noise seed")
        ax.set_ylabel("Recovery fraction")
        ax.set_title("Noise robustness")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.show()

    if len(ncw_df):
        fig, ax1 = plt.subplots(figsize=(7, 4))
        ax1.plot(ncw_df["N_CW"], ncw_df["sweep_frac"], marker="o", label="multi-sweep")
        ax1.plot(ncw_df["N_CW"], ncw_df["polished_frac"], marker="o", label="polished")
        ax1.set_ylim(0, 1.05)
        ax1.set_xlabel("N_CW")
        ax1.set_ylabel("Recovery fraction")
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc="lower right")
        ax2 = ax1.twinx()
        ax2.plot(ncw_df["N_CW"], ncw_df["contrast_median"], marker="s", color="tab:red", label="median contrast")
        ax2.set_ylabel("Median contrast")
        ax2.legend(loc="upper left")
        ax1.set_title("N_CW scaling")
        plt.show()
else:
    print("Plots skipped")